# Simple 2D diffusion FDM example: source at the center

This notebook is a compact teaching example for the **2D diffusion equation**

$$
\frac{\partial c}{\partial t}
=
D\left(
\frac{\partial^2 c}{\partial x^2}
+
\frac{\partial^2 c}{\partial y^2}
\right),
$$

solved by a **finite-difference method (FDM)** on a square computational box.

We will:

1. define a 2D grid,
2. place an initial source at the center,
3. update the field using an explicit finite-difference scheme,
4. visualize how the source spreads with time.

This is a useful starting point for later materials-science topics such as
diffusion, heat conduction, and phase-field modeling.

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. The 2D diffusion equation

We solve

$$
\frac{\partial c}{\partial t}
=
D\left(
\frac{\partial^2 c}{\partial x^2}
+
\frac{\partial^2 c}{\partial y^2}
\right).
$$

Using central differences in space and forward Euler in time, the update rule is

$$
c_{i,j}^{n+1}
=
c_{i,j}^{n}
+
D\Delta t
\left[
\frac{c_{i+1,j}^{n}-2c_{i,j}^{n}+c_{i-1,j}^{n}}{\Delta x^2}
+
\frac{c_{i,j+1}^{n}-2c_{i,j}^{n}+c_{i,j-1}^{n}}{\Delta y^2}
\right].
$$

For the special case \(\Delta x = \Delta y\), this becomes

$$
c_{i,j}^{n+1}
=
c_{i,j}^{n}
+
\alpha
\left(
c_{i+1,j}^{n}+c_{i-1,j}^{n}+c_{i,j+1}^{n}+c_{i,j-1}^{n}-4c_{i,j}^{n}
\right),
$$

where

$$
\alpha = \frac{D\Delta t}{\Delta x^2}.
$$

For stability in 2D with this explicit scheme, we need roughly

$$
\alpha \le \frac{1}{4}.
$$

## 3. Set up the computational grid

In [ ]:
# Domain and grid
Lx = 1.0
Ly = 1.0
Nx = 81
Ny = 81

x = np.linspace(0, Lx, Nx)
y = np.linspace(0, Ly, Ny)

dx = x[1] - x[0]
dy = y[1] - y[0]

X, Y = np.meshgrid(x, y, indexing='ij')

print("dx =", dx)
print("dy =", dy)

## 4. Material and time-stepping parameters

In [ ]:
D = 1.0e-3

# Choose dt using the 2D explicit stability guideline
dt_stable = dx**2 / (4 * D)
dt = 0.8 * dt_stable   # slightly below the stability limit

alpha_x = D * dt / dx**2
alpha_y = D * dt / dy**2

print("dt_stable =", dt_stable)
print("dt =", dt)
print("alpha_x =", alpha_x)
print("alpha_y =", alpha_y)
print("alpha_x + alpha_y =", alpha_x + alpha_y)

For a general rectangular grid, the explicit stability condition is

$$
\frac{D\Delta t}{\Delta x^2} + \frac{D\Delta t}{\Delta y^2} \le \frac{1}{2}.
$$

Here we choose $\Delta t$ to satisfy that condition.

## 5. Initial condition: source at the center

We start with zero concentration everywhere except at the center of the box.

This gives a point-like initial source on the grid.

In [ ]:
c0 = np.zeros((Nx, Ny))

ic = Nx // 2
jc = Ny // 2

c0[ic, jc] = 1.0

plt.figure(figsize=(6,5))
plt.imshow(c0.T, origin='lower', extent=[0, Lx, 0, Ly], aspect='auto')
plt.colorbar(label='c')
plt.title("Initial condition: source at the center")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

## 6. Boundary conditions

For simplicity, we use **zero Dirichlet boundary conditions**:

$$
c = 0
\quad \text{on the boundary}.
$$

That means the edges of the box are kept at zero concentration.

This is easy to implement and good for a first teaching example.

## 7. One explicit FDM update step

In [ ]:
def diffusion_step(c, D, dt, dx, dy):
    c_new = c.copy()

    c_new[1:-1, 1:-1] = (
        c[1:-1, 1:-1]
        + D * dt * (
            (c[2:, 1:-1] - 2*c[1:-1, 1:-1] + c[:-2, 1:-1]) / dx**2
            + (c[1:-1, 2:] - 2*c[1:-1, 1:-1] + c[1:-1, :-2]) / dy**2
        )
    )

    # Dirichlet boundary conditions: keep edges at zero
    c_new[0, :] = 0.0
    c_new[-1, :] = 0.0
    c_new[:, 0] = 0.0
    c_new[:, -1] = 0.0

    return c_new

## 8. March forward in time

In [ ]:
nsteps = 800
save_steps = [0, 10, 50, 100, 200, 400, 800]

snapshots = {}
c = c0.copy()
snapshots[0] = c.copy()

for n in range(1, nsteps + 1):
    c = diffusion_step(c, D, dt, dx, dy)
    if n in save_steps:
        snapshots[n] = c.copy()

print("Saved snapshots at steps:", sorted(snapshots.keys()))

## 9. Plot the 2D concentration field at selected times

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6), constrained_layout=True)

for ax, n in zip(axes.flat, save_steps[:6]):
    im = ax.imshow(
        snapshots[n].T,
        origin='lower',
        extent=[0, Lx, 0, Ly],
        aspect='auto'
    )
    ax.set_title(f"step = {n}, t = {n*dt:.4f}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    fig.colorbar(im, ax=ax, shrink=0.85)

plt.show()

You should see the initial source spread out symmetrically from the center.

## 10. Final state

In [ ]:
plt.figure(figsize=(6,5))
plt.imshow(
    snapshots[800].T,
    origin='lower',
    extent=[0, Lx, 0, Ly],
    aspect='auto'
)
plt.colorbar(label='c')
plt.title(f"Final concentration field at t = {800*dt:.4f}")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

## 11. Cross section through the center

A useful way to inspect the diffusion profile is to look at a line cut through the center.

In [ ]:
plt.figure(figsize=(7,5))

for n in [0, 10, 50, 100, 200, 400, 800]:
    plt.plot(x, snapshots[n][:, jc], label=f"step={n}")

plt.xlabel("x")
plt.ylabel("c(x, y_center)")
plt.title("Center-line concentration profiles")
plt.legend()
plt.show()

## 12. Check total mass in the box

Because we are using zero Dirichlet boundaries, material can effectively diffuse out through the fixed-zero edges.
So the total mass in the computational box will generally decrease with time.

The discrete total mass is approximately

$$
M(t) \approx \sum_{i,j} c_{i,j} \, \Delta x \, \Delta y.
$$

In [ ]:
mass_history = []

c = c0.copy()
mass_history.append(np.sum(c) * dx * dy)

for n in range(1, nsteps + 1):
    c = diffusion_step(c, D, dt, dx, dy)
    mass_history.append(np.sum(c) * dx * dy)

tvals = np.arange(nsteps + 1) * dt

plt.figure(figsize=(7,5))
plt.plot(tvals, mass_history)
plt.xlabel("t")
plt.ylabel("total mass in box")
plt.title("Mass history with zero Dirichlet boundaries")
plt.show()

print("Initial mass =", mass_history[0])
print("Final mass   =", mass_history[-1])

## 13. Optional extension: zero-flux boundaries

If you want to conserve mass more naturally, a common alternative is the **zero-flux (Neumann)** condition

$$
\frac{\partial c}{\partial n} = 0
$$

on the boundary.

That would require a slightly different treatment of the edge nodes.
For a first notebook, the zero-Dirichlet version is simpler.

## 14. A reusable simulation function

In [ ]:
def run_diffusion_2d_center_source(
    Lx=1.0, Ly=1.0, Nx=81, Ny=81, D=1.0e-3, nsteps=800, source_value=1.0
):
    x = np.linspace(0, Lx, Nx)
    y = np.linspace(0, Ly, Ny)
    dx = x[1] - x[0]
    dy = y[1] - y[0]

    dt = 0.8 / (2 * D * (1/dx**2 + 1/dy**2))  # stable explicit step

    c = np.zeros((Nx, Ny))
    ic = Nx // 2
    jc = Ny // 2
    c[ic, jc] = source_value

    for _ in range(nsteps):
        c = diffusion_step(c, D, dt, dx, dy)

    return x, y, c, dt

x2, y2, c2, dt2 = run_diffusion_2d_center_source()
print("Returned field shape:", c2.shape)
print("dt used:", dt2)

## 15. Summary

In this notebook, we:

- solved the 2D diffusion equation with an explicit finite-difference method,
- placed an initial source at the center of the computational box,
- visualized the spreading concentration field,
- examined center-line profiles,
- monitored the total mass in the box.

This is a good minimal example for introducing 2D transport on a grid.

## 16. Optional exercises

1. Replace the point source with a small Gaussian initial condition.
2. Change the boundary condition to zero-flux and compare the mass history.
3. Change the diffusion coefficient \(D\) and observe how the spreading rate changes.
4. Animate the field as it evolves in time.
5. Use a rectangular box with different \(L_x\) and \(L_y\).